In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import time
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64
batch_size = 128

train_split = 0.8
max_iterations = 3000
learning_rate = 3e-4
eval_iters = 500
dropout = 0.2
n_embd = 384
n_layer = 8
n_head = 8

cuda


In [15]:
"""
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))
print(chars)
print(len(chars))
vocab_size = len(chars)
"""

dataset = load_dataset(
    "Skylion007/openwebtext",
    "plain_text",
    split="train",
    streaming=True
)

# Take a small amount of OpenWebText for now
texts = []

for example in dataset.take(10000):
    texts.append(example["text"])

text = "\n".join(texts)

print(f"Loaded {len(text):,} characters")
print(text[:1000])

chars = sorted(set(text))
print(chars)
print(len(chars))
vocab_size = len(chars)

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loaded 48,604,443 characters
Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital Friday night after a Belgian medical team evacuated the area, saying it was concerned about security.

The decision left CNN Chief Medical Correspondent Sanjay Gupta as the only doctor at the hospital to get the patients through the night.

CNN initially reported, based on conversations with some of the doctors, that the United Nations ordered the Belgian First Aid and Support Team to evacuate. However, Belgian Chief Coordinator Geert Gijs, a doctor who was at the hospital with 60 Belgian medical personnel, said it was his decision to pull the team out for the night. Gijs said he requested U.N. security personnel to staff the hospital overnight, but was told that peacekeepers would only be able to evacuate the team.

He said it was a "tough decision" but that he accepted the U.N. offer to evacuate after a Canad

# Tokenizer

In [16]:
string_to_int = { c:i for i,c in enumerate(chars) }
int_to_string = { i:c for i,c in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([49, 80, 83, 85, 14, 66, 86, 14, 49, 83, 74, 79, 68, 70, 13,  1, 41, 66,
        74, 85, 74,  1,  9, 36, 47, 47, 10,  1, 14, 14,  1, 38, 66, 83, 85, 73,
        82, 86, 66, 76, 70,  1, 87, 74, 68, 85, 74, 78, 84, 13,  1, 88, 83, 74,
        85, 73, 74, 79, 72,  1, 74, 79,  1, 81, 66, 74, 79,  1, 66, 79, 69,  1,
        72, 83, 66, 84, 81, 74, 79, 72,  1, 66, 85,  1, 77, 74, 71, 70, 13,  1,
        88, 66, 85, 68, 73, 70, 69,  1, 69, 80])


# Splits

In [17]:
n = int(train_split*len(data))

train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch("train")
print('inputs: ')
print(x)
print('targets: ')
print(y)

inputs: 
tensor([[ 0, 52, 85,  ...,  1, 85, 73],
        [72,  1, 68,  ...,  1, 77, 74],
        [53, 48,  1,  ..., 40, 83, 66],
        ...,
        [66, 85,  1,  ..., 68, 83, 70],
        [66, 85,  1,  ..., 86, 66, 77],
        [41, 86, 78,  ...,  1, 68, 66]], device='cuda:0')
targets: 
tensor([[52, 85, 66,  ..., 85, 73, 70],
        [ 1, 68, 86,  ..., 77, 74, 76],
        [48,  1, 45,  ..., 83, 66, 73],
        ...,
        [85,  1, 73,  ..., 83, 70, 66],
        [85,  1, 77,  ..., 66, 77, 77],
        [86, 78, 66,  ..., 68, 66, 79]], device='cuda:0')


In [18]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('When input is', context, 'target is', target)

When input is tensor([49]) target is tensor(80)
When input is tensor([49, 80]) target is tensor(83)
When input is tensor([49, 80, 83]) target is tensor(85)
When input is tensor([49, 80, 83, 85]) target is tensor(14)
When input is tensor([49, 80, 83, 85, 14]) target is tensor(66)
When input is tensor([49, 80, 83, 85, 14, 66]) target is tensor(86)
When input is tensor([49, 80, 83, 85, 14, 66, 86]) target is tensor(14)
When input is tensor([49, 80, 83, 85, 14, 66, 86, 14]) target is tensor(49)
When input is tensor([49, 80, 83, 85, 14, 66, 86, 14, 49]) target is tensor(83)
When input is tensor([49, 80, 83, 85, 14, 66, 86, 14, 49, 83]) target is tensor(74)
When input is tensor([49, 80, 83, 85, 14, 66, 86, 14, 49, 83, 74]) target is tensor(79)
When input is tensor([49, 80, 83, 85, 14, 66, 86, 14, 49, 83, 74, 79]) target is tensor(68)
When input is tensor([49, 80, 83, 85, 14, 66, 86, 14, 49, 83, 74, 79, 68]) target is tensor(70)
When input is tensor([49, 80, 83, 85, 14, 66, 86, 14, 49, 83, 74

# Model

In [19]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y) # calls forward
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [20]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        #input and output of size (B, T, C)
        B, T, C = x.shape
        k = self.key(x) #--> (B, T, head_size) as per the Linear transformation of self.key
        q = self.query(x) #--> (B, T, head_size) as per the Linear transformation of self.query
        
        #attention scores
        wei = q @ k.transpose(-2, -1) / k.shape[-1]**0.5 #(B, T, head_size) @ (B, head_size, T) --> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #(B, T, T)
        wei = F.softmax(wei, dim=-1) #(B, T, T)
        wei = self.dropout(wei)

        #weighted aggregation of the values
        v = self.value(x) #(B, T, head_size)
        out = wei @ v #(B, T, T) @ (B, T, head_size) --> (B, T, head_size)
        return out
        

In [21]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) #(B, T, F) --> (B, T, [head1, head1, head1, head1, head2, head2, head2, head2, ...])
        out = self.dropout(self.proj(out))
        return out

In [22]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [23]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head #decides how many features each head is going to capture
        self.sa = MultiHeadAttention(n_head, head_size) #self-attention
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        
        return x

In [24]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        #index and targets are (B, T) sized tensors of integers
        #logits = self.token_embedding_table(index)
        B, T = index.shape

        tok_emb = self.token_embedding_table(index) #(B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T, C)
        x = tok_emb + pos_emb #(B, T, C)
        x = self.blocks(x) #(B, T, C)
        x = self.ln_f(x) #(B, T, C)
        logits = self.lm_head(x) #(B, T, vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, loss = self.forward(index_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        
        return index

model = GPTLanguageModel(vocab_size)
m = model.to(device)

#context = torch.zeros((1,1), dtype=torch.long, device = device)
#generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
#print(generated_chars)

# Optimization

In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iterations):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
    
    xb, yb = get_batch('train')

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step: 0, train loss: 7.470, val loss: 7.470
step: 500, train loss: 1.900, val loss: 1.929
step: 1000, train loss: 1.678, val loss: 1.711
step: 1500, train loss: 1.590, val loss: 1.626
step: 2000, train loss: 1.540, val loss: 1.574
step: 2500, train loss: 1.510, val loss: 1.551
1.4768787622451782


In [13]:
context = torch.zeros((1,1), dtype=torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


EU/SW Disprision Datelicra as a point.

French thenior not atvertuits win that, reporting to the carry make promother asnowled Southern High Bolifors for the case,

In Bomninstra of letari belief on cells over the pubt against Report Dakis oh No Dismant, into a former merging on a Babdaria’s First Lovine alternion Clocals.

While Roid Thiurzing safel asked as for Alroni Vealthi would be presidently member hero. Mugwell areas Stephode enited Tursard, but too a year-old's southorer, a hope wanted 
